# CHATBOT CIMA — Actividad 6
## Evaluación, validación y ajuste de decisiones

Este notebook evalúa los modelos congelados de la Actividad 5 y produce evidencia reproducible para el reporte y el tablero Trello.

**Flujo:** matrices de confusión → métricas y ROC/AUC → baseline → validación cruzada OOF de cuatro pliegues → umbral derivado de OOF → comparación estadística pareada → simulación offline de políticas A/B.

> **Límite de interpretación:** el dataset es sintético; la partición de prueba tiene 26 registros y solo 2 casos `AMBIGUA`. Los resultados son exploratorios y no autorizan el uso productivo ni validaciones de pagos.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alfredosalazar-ai/deep-learning-experiments/blob/main/Actividad6/fuentes/CIMA_Actividad6_Evaluacion_Validacion.ipynb)


## 1. Preparar el entorno y ubicar el proyecto

La versión pública del proyecto se obtiene desde GitHub. Mientras la carpeta `Actividad6/` aún no esté publicada, se puede cargar el ZIP de preparación de esta actividad. No se necesita instalar nada en la computadora local.

In [ ]:
from pathlib import Path
import subprocess, sys, zipfile, shutil, os
from IPython.display import display, Image

REPO_URL = "https://github.com/alfredosalazar-ai/deep-learning-experiments.git"
REPO_DIR = Path("/content/deep-learning-experiments")

# Clonar o actualizar el repositorio público, si está disponible.
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

PROJECT_ROOT = REPO_DIR / "Actividad6"

# Fallback para la primera ejecución: se puede subir el ZIP del proyecto A6.
if not (PROJECT_ROOT / "fuentes" / "evaluacion.py").is_file():
    from google.colab import files
    print("Actividad6 aún no aparece en la rama main. Sube CIMA_Actividad6_GitHub.zip.")
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    assert zip_name, "Debes cargar el ZIP que contiene la carpeta Actividad6."
    fallback = Path("/content/actividad6_cargada")
    fallback.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(fallback)
    candidates = list(fallback.rglob("evaluacion.py"))
    assert candidates, "El ZIP no contiene fuentes/evaluacion.py."
    PROJECT_ROOT = candidates[0].parent.parent

assert (PROJECT_ROOT / "fuentes" / "evaluacion.py").is_file(), f"No se encontró evaluacion.py en {PROJECT_ROOT}"
print("Proyecto:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

## 2. Instalar dependencias compatibles

Se fija scikit-learn 1.6.1 porque los modelos `.joblib` se serializaron con esa versión en la Actividad 5. Esto reduce los riesgos de incompatibilidad al cargar objetos persistidos.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)

import sklearn, numpy, pandas, scipy
print("scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
assert sklearn.__version__ == "1.6.1", "Se esperaba scikit-learn 1.6.1 para compatibilidad con los modelos de A5."

## 3. Ejecutar la evaluación

El script conserva los modelos originales y no modifica las particiones. Calcula la evaluación de prueba con los modelos A5, y estima umbrales mediante puntuaciones *out-of-fold* con validación cruzada OOF únicamente en entrenamiento. La prueba final no se utiliza para elegir umbrales.

In [ ]:
resultado = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "fuentes" / "evaluacion.py")],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(resultado.stdout)
if resultado.stderr:
    print("Mensajes del proceso:\n", resultado.stderr[-5000:])
if resultado.returncode != 0:
    raise RuntimeError(f"La evaluación terminó con código {resultado.returncode}")

## 4. Consultar métricas, umbrales y significancia

In [ ]:
import pandas as pd
OUT = PROJECT_ROOT / "resultados"

def mostrar_csv(nombre):
    path = OUT / nombre
    assert path.is_file(), f"No se generó {path}"
    print(f"\n### {nombre}")
    display(pd.read_csv(path, encoding="utf-8-sig"))

for nombre in [
    "metricas_actividad6.csv",
    "validacion_cruzada.csv",
    "analisis_umbrales.csv",
    "metricas_por_clase.csv",
    "comparacion_estadistica.csv",
    "simulacion_ab_politicas.csv",
]:
    mostrar_csv(nombre)

## 5. Visualizaciones generadas

In [ ]:
from IPython.display import display, Image
fig_dir = OUT / "graficas"
for img in sorted(fig_dir.glob("*.png")):
    print(img.name)
    display(Image(filename=str(img), width=760))

## 6. Exportar resultados para GitHub y el ZIP académico

El ZIP solo contiene los resultados generados por esta ejecución. Al cierre se integrarán a `Actividad6/resultados/` y se versionarán junto con el notebook y el informe técnico. La simulación A/B es retrospectiva y pareada; no constituye una prueba aleatoria con usuarios reales.

In [ ]:
from google.colab import files
import shutil

archivo_zip_resultados = "/content/CIMA_Actividad6_resultados.zip"
shutil.make_archive(archivo_zip_resultados[:-4], "zip", OUT)
print("Paquete generado:", archivo_zip_resultados)
files.download(archivo_zip_resultados)

## 7. Criterios de interpretación y control

- No interpretar el resultado perfecto observado anteriormente en 26 consultas sintéticas como precisión productiva.
- La puntuación de `LinearSVC` es un margen de decisión, no una probabilidad calibrada.
- El umbral se selecciona con OOF y debe evaluarse después sobre la prueba reservada.
- Las pruebas estadísticas sobre 26 casos tienen baja potencia; un valor p no demuestra equivalencia ni seguridad operativa.
- La comparación A/B se limita a una simulación offline sobre los mismos casos, no a un experimento real con usuarios.
- El modelo puede enrutar una consulta, pero no verifica folios, pagos, saldos ni estado de asignación. Esas validaciones deben continuar mediante reglas deterministas y controles humanos.